# MonoDGP M57: portable deformable-attention parity

This notebook adds one opt-in rank-five `grid_sample` implementation to the pinned MonoDGP graph. It performs no training or Core ML conversion. Use a CUDA GPU and stop after the manifest and smoke report are produced.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
import json, os, shlex, shutil, subprocess, sys
MOBILE_REPO=Path('/content/mobile_adas3d')
MONODGP_REPO=Path('/content/MonoDGP_M57')
MONODGP_COMMIT='aa059a18214aebf644510e7f0793971b403f9d14'
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT=Path('/content/kitti')
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
DATASET_ROOT=Path('/content/monodgp_kitti_m57')
M56D_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m56d_det2d_fp32_storage')
M56D_MANIFEST=M56D_ROOT/'m56d_compression_manifest.json'
M56D_SMOKE=M56D_ROOT/'m56d_det2d_fp32_smoke.json'
M56D_GATE=M56D_ROOT/'complete_evaluation/m56d_det2d_fp32_gate.json'
M56D_COMPARISON=M56D_ROOT/'complete_evaluation/m56d_det2d_fp32_comparison.csv'
OUTPUT_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m57_deformable_attention')
LOG_DIR=OUTPUT_ROOT/'colab_logs'
def run(command,cwd=None,env=None):
    command=[str(item) for item in command]; print('+',shlex.join(command),flush=True)
    merged=os.environ.copy(); merged.update(env or {})
    result=subprocess.run(command,cwd=cwd,env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
def run_logged(command,cwd,log_path,env=None):
    command=[str(item) for item in command]; print('+',shlex.join(command),flush=True)
    log_path=Path(log_path); log_path.parent.mkdir(parents=True,exist_ok=True)
    merged=os.environ.copy(); merged.update(env or {}); tail=deque(maxlen=100)
    with log_path.open('w',encoding='utf-8') as log:
        process=subprocess.Popen(command,cwd=cwd,env=merged,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
        code=process.wait()
    if code: raise RuntimeError(f'Exit {code}; full log={log_path}\n'+'\n'.join(tail))
    return log_path
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
run(['nvidia-smi'])

In [ ]:
# Fetch exact sources, apply audited patches, and build the native reference extension.
if not MOBILE_REPO.exists():
    run(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO])
else:
    run(['git','pull','--ff-only'],cwd=MOBILE_REPO)
if not MONODGP_REPO.exists():
    run(['git','clone','https://github.com/PuFanqi23/MonoDGP.git',MONODGP_REPO])
run(['git','fetch','--all'],cwd=MONODGP_REPO)
run(['git','checkout',MONODGP_COMMIT],cwd=MONODGP_REPO)
run([sys.executable,'-m','pip','install','-q','pyyaml','scipy','opencv-python-headless','numba','scikit-image','scikit-learn','tqdm','ninja','pandas'])
run([sys.executable,'scripts/patch_monodgp_colab_compat.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
run([sys.executable,'scripts/patch_monodgp_m54_training.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
run([sys.executable,'scripts/patch_monodgp_m57_deformable_attention.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
changed=set(subprocess.run(['git','diff','--name-only'],cwd=MONODGP_REPO,check=True,capture_output=True,text=True).stdout.splitlines())
expected={
    'lib/datasets/kitti/kitti_dataset.py',
    'lib/helpers/save_helper.py',
    'lib/helpers/trainer_helper.py',
    'lib/models/monodgp/ops/modules/ms_deform_attn.py',
    'lib/models/monodgp/ops/setup.py',
    'lib/models/monodgp/ops/src/cuda/ms_deform_attn_cuda.cu',
    'tools/train_val.py',
}
if changed != expected: raise RuntimeError(f'Unexpected patched source set: {changed}')
ops=MONODGP_REPO/'lib/models/monodgp/ops'
shutil.rmtree(ops/'build',ignore_errors=True)
run([sys.executable,'setup.py','build','install'],cwd=ops,env={'MAX_JOBS':'2'})
run([sys.executable,'-c','import torch, MultiScaleDeformableAttention; print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0))'],cwd=MONODGP_REPO)

In [ ]:
# Create the canonical Chen-split KITTI view without copying images.
def resolve(root,names):
    for name in names:
        path=root/name
        if path.is_dir(): return path
sources={
    key:resolve(LOCAL_DATASET_ROOT,names) or resolve(DRIVE_DATASET_ROOT,names)
    for key,names in {
        'image_2':['training/image_2','training/image_02'],
        'label_2':['training/label_2','training/label_02'],
        'calib':['training/calib'],
    }.items()
}
if any(path is None for path in sources.values()): raise FileNotFoundError(sources)
(DATASET_ROOT/'training').mkdir(parents=True,exist_ok=True)
(DATASET_ROOT/'ImageSets').mkdir(parents=True,exist_ok=True)
for name,target in sources.items():
    link=DATASET_ROOT/'training'/name
    if link.is_symlink() and link.resolve()==target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target,target_is_directory=True)
for split in ('train','val'):
    shutil.copy2(SPLIT_DIR/f'{split}.txt',DATASET_ROOT/'ImageSets'/f'{split}.txt')
assert len((DATASET_ROOT/'ImageSets/train.txt').read_text().splitlines())==3712
assert len((DATASET_ROOT/'ImageSets/val.txt').read_text().splitlines())==3769
for required in (M56D_MANIFEST,M56D_SMOKE,M56D_GATE,M56D_COMPARISON):
    if not required.is_file(): raise FileNotFoundError(required)

In [ ]:
# Freeze exact M56d provenance, patched source, module inventory, and native-default behavior.
os.environ.pop('MONODGP_PORTABLE_DEFORM_ATTN',None)
PREPARE_LOG=run_logged([
    sys.executable,'-u','scripts/prepare_monodgp_m57_deformable_attention.py',
    '--monodgp-repo',MONODGP_REPO,
    '--m56d-manifest',M56D_MANIFEST,
    '--m56d-smoke',M56D_SMOKE,
    '--m56d-gate',M56D_GATE,
    '--m56d-comparison',M56D_COMPARISON,
    '--output-root',OUTPUT_ROOT,
],MOBILE_REPO,LOG_DIR/'m57_prepare.log')
MANIFEST=OUTPUT_ROOT/'m57_deformable_attention_manifest.json'
manifest=json.loads(MANIFEST.read_text())
assert manifest['smoke_authorized'] and all(manifest['preparation_gate_results'].values())
assert manifest['training_performed'] is False and manifest['weights_changed'] is False
assert manifest['native_path_default'] is True
print('M57 preparation gates:',json.dumps(manifest['preparation_gate_results'],indent=2))

In [ ]:
# Real CUDA barrier: nine local comparisons, full-model parity, trace audit, and 5/100 timing per path.
SMOKE=OUTPUT_ROOT/'m57_deformable_attention_smoke.json'
SMOKE_LOG=run_logged([
    sys.executable,'-u','scripts/smoke_test_monodgp_m57_deformable_attention.py',
    '--monodgp-repo',MONODGP_REPO,
    '--manifest',MANIFEST,
    '--output',SMOKE,
],MOBILE_REPO,LOG_DIR/'m57_smoke.log')
smoke=json.loads(SMOKE.read_text())
assert smoke['all_smoke_gates_passed'] and smoke['full_evaluation_authorized']
print('M57 gates:',json.dumps(smoke['gate_results'],indent=2))
print('Module parity:',json.dumps(smoke['module_parity'],indent=2))
print('Full-output parity:',json.dumps(smoke['raw_output_parity'],indent=2))
print('Native/portable timing:',json.dumps(smoke['runtime_comparison'],indent=2))

In [ ]:
# Stop point 1: do not run full validation or Core ML conversion yet.
print('Stop point 1 reached. Return these exact files for review:')
print(MANIFEST)
print(SMOKE)
print('Full evaluation remains review-locked; direct Core ML conversion is unauthorized.')